# 02 — GraphRAG: Multi-hop Reasoning and Provenance

**Track:** Advanced · **Stage:** Advanced Retrieval

Standard vector retrieval struggles with **multi-hop** questions. For example: *"Does the supplier of Project Atlas need to comply with Regulation R-17?"* 

If Document A says "Acme supplies Atlas" and Document B says "Acme must comply with R-17", a standard vector search for "Atlas R-17 compliance" might miss both documents because neither document individually matches the query semantically.

**GraphRAG** solves this by explicitly extracting entities and relationships into a Knowledge Graph (e.g., Neo4j, NetworkX) *before* retrieval. During retrieval, we traverse the graph to collect a bounded neighborhood of explicitly connected facts.

In this comprehensive deep dive, we will:
1. **Part 1: The Theory.** Manually build a small property graph to understand how provenance and entity resolution work.
2. **Part 2: Production Implementation.** Use **LangChain** and **NetworkX** to simulate LLM triplet extraction and perform shortest-path graph traversals.

---
## Part 1: The Theory of GraphRAG

Before we introduce complex frameworks, let's understand the core concept: a `Fact` is just a typed edge `(Subject -> Predicate -> Object)` with some metadata (provenance).

In [ ]:
from collections import namedtuple

# A Fact represents an Edge in our graph
Fact = namedtuple("Fact", ["id", "subject", "predicate", "object", "source"])

# 1. Define our manual Knowledge Graph
facts = [
    Fact("f1", "Project Atlas", "depends_on", "VectorDB-X", "architecture.md"),
    Fact("f2", "VectorDB-X", "supplied_by", "Acme Systems", "vendor_list.csv"),
    Fact("f3", "Acme Systems", "must_certify", "Regulation R-17", "compliance.md")
]

# 2. Graph Traversal logic (simulating a hop)
def traverse(start_entity: str, hops: int = 2):
    print(f"Traversing starting from: '{start_entity}'")
    current_entities = {start_entity}
    retrieved_facts = []
    
    for hop in range(hops):
        next_entities = set()
        for fact in facts:
            if fact.subject in current_entities:
                retrieved_facts.append(fact)
                next_entities.add(fact.object)
            elif fact.object in current_entities:
                retrieved_facts.append(fact)
                next_entities.add(fact.subject)
        current_entities = next_entities
    
    return retrieved_facts

print("--- Manual Graph Traversal ---")
found_facts = traverse("Project Atlas", hops=3)
for f in set(found_facts): # Use set to remove duplicates from bidirectional search
    print(f"Found Fact: {f.subject} -> {f.predicate} -> {f.object} (Source: {f.source})")

---
## Part 2: Production Implementation with NetworkX

In a production system, you do not write facts manually. You use an LLM (like LangChain's `LLMGraphTransformer`) to extract triplets from unstructured text, and store them in a graph database (like Neo4j) or an in-memory graph (like NetworkX).

In [ ]:
# !pip install langchain langchain-community networkx matplotlib

import networkx as nx
from langchain_core.documents import Document
from langchain_community.llms.fake import FakeListLLM
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

### Step A: Load Triplets into NetworkX

We will mock the LLM extraction step and directly build a NetworkX directed graph.

In [ ]:
# Simulated output from an LLM Graph Extractor (e.g. LLMGraphTransformer)
extracted_triplets = [
    ("Project Atlas", "DEPENDS_ON", "VectorDB-X"),
    ("VectorDB-X", "SUPPLIED_BY", "Acme Systems Inc"),
    ("Acme Systems Inc", "MUST_COMPLY_WITH", "Regulation R-17")
]

# Build the local NetworkX graph
G = nx.DiGraph()
for subject, relation, target in extracted_triplets:
    G.add_edge(subject, target, label=relation)

print("Knowledge Graph Built.")
print("Nodes:", G.nodes())
print("Edges:", G.edges(data=True))

### Step B: The Multi-Hop Query

When a user asks a complex question, we use an LLM to extract the two extreme entities from the question. Then we use NetworkX to find the shortest path connecting them.

In [ ]:
question = "Does the supplier of Project Atlas need to comply with Regulation R-17?"

# 1. LLM Entity Extraction (Mocked)
query_entities = ["Project Atlas", "Regulation R-17"]

# 2. Graph Traversal: Find the path connecting them
try:
    # NetworkX shortest path converts the directed graph to undirected for traversal
    path = nx.shortest_path(G.to_undirected(), source=query_entities[0], target=query_entities[1])
    print(f"\nPath found in Knowledge Graph: {' -> '.join(path)}")
    
    # 3. Retrieve the context along that path
    context_statements = []
    for i in range(len(path) - 1):
        n1, n2 = path[i], path[i+1]
        # Check direction to print the correct relationship
        if G.has_edge(n1, n2):
            rel = G[n1][n2]['label']
            context_statements.append(f"{n1} {rel} {n2}")
        else:
            rel = G[n2][n1]['label']
            context_statements.append(f"{n2} {rel} {n1}")
            
    print("\nExtracted Context for LLM:")
    for stmt in context_statements:
        print(f"- {stmt}")
        
except nx.NetworkXNoPath:
    print("No path found connecting the entities.")

### Step C: Generate the Grounded Answer

Now we pass this highly dense, exact context to the LLM to generate the final answer.

In [ ]:
prompt = ChatPromptTemplate.from_template(
    "Answer the question using ONLY the provided graph context.\n"
    "Context:\n{context}\n\n"
    "Question: {question}"
)

mock_llm = FakeListLLM(responses=[
    "Yes, Project Atlas depends on VectorDB-X, which is supplied by Acme Systems Inc. "
    "According to the graph, Acme Systems Inc must comply with Regulation R-17."
])

chain = prompt | mock_llm | StrOutputParser()

final_answer = chain.invoke({
    "context": "\n".join(context_statements), 
    "question": question
})
print(f"\nFINAL ANSWER:\n{final_answer}")

## Reflection

1. **Cost of GraphRAG:** Extracting triplets from 10,000 documents requires passing all 10,000 documents through an LLM *at ingestion time*. This is extremely expensive compared to fast embedding vectors. GraphRAG should be reserved for highly interconnected, high-value datasets.
2. **Hybrid Graph+Vector:** Modern architectures (like Microsoft's GraphRAG) often embed the *nodes* and *descriptions* in a vector database, allowing you to do semantic search to find the starting node, and then graph traversal to find the neighborhood.